# **CNN yordamida tasvirlarni tasniflash**

## Ushbu noutbukda

## MNIST ma'lumotlar to'plami yordamida

## qo'lda yozilgan raqamlarni

## tasniflash uchun

## Birlashtirilgan Neyron Tarmoq (CNN)

## yaratilishi va shug'ullantirishni o'rganamiz

In [99]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

# torch: Neyron tarmoqlar uchun asosiy kutubxona, tensor operatsiyalari va avtomatik differensiallash (avtograd)ni ta'minlaydi.
# torch.nn: Neyron tarmoqlar qatlamlari, yo'qotish funksiyalari va faollashish funksiyalarini yaratish uchun ishlatiladi.
# torch.optim: Model parametrlari uchun optimallashtirish algoritmlarini (masalan, SGD, Adam) o'z ichiga oladi.
# torchvision: Kompyuter ko'rishi vazifalari uchun ma'lumotlar to'plamlari va oldindan tayyorlangan modellar bilan ishlaydi.
# torchvision.transforms: Tasvirlarni oldindan qayta ishlash (preprocessing) va ma'lumotlarni kengaytirish (data augmentation) uchun transformatsiyalar taqdim etadi.


In [100]:
torch.cuda.is_available()

# funksiyasi PyTorch'ning grafik ishlov berish birligi (GPU) ya'ni CUDA'dan foydalana olish imkoniyati
# bor-yo'qligini tekshiradi. Agar qurilmada CUDA qo'llab-quvvatlansa, True qaytaradi, aks holda False qaytaradi.

True

In [101]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# kod device nomli o'zgaruvchini belgilaydi. Agar qurilmada CUDA (ya'ni, NVIDIA GPU) mavjud bo'lsa,
# u device='cuda' qilib o'rnatiladi, aks holda device='cpu' (markaziy protsessor) qilib o'rnatiladi.
# Bu modelni tezroq hisoblash uchun GPU'dan foydalanish imkoniyatini beradi.

In [102]:
transforms = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# transforms.Compose: Bir nechta tasvir o'zgartirishlarni ketma-ket bajarish uchun.
# transforms.ToTensor(): Tasvirni PyTorch tensoriga aylantiradi va piksel qiymatlarini [0, 1] oralig'iga normalizatsiya qiladi.

In [103]:
# MNIST
train_date = torchvision.datasets.MNIST(
     root='./data',
     train=True,
     download=True,
     transform=transforms
)

test_date = torchvision.datasets.MNIST(
     root='./data',
     train=False,
     download=True,
     transform=transforms
)

# torchvision.datasets.MNIST(...): Bu funksiya qo'lda yozilgan raqamlar tasvirlaridan
# iborat mashhur MNIST ma'lumotlar to'plamini yuklash uchun ishlatiladi.
# root='./data': Ma'lumotlar to'plami saqlanadigan joyni (papka) belgilaydi.
# train=True: O'quv (training) to'plamini yuklaydi. train=False esa test to'plamini yuklaydi.
# download=True: Agar ma'lumotlar to'plami mahalliy ravishda mavjud bo'lmasa, uni internetdan yuklab oladi.
# transform=transforms: Yuqorida aniqlangan transforms ob'ektini ma'lumotlar to'plamidagi har bir
# tasvirga qo'llaydi, masalan, uni tensorga aylantirish uchun.


In [104]:
train_loader = torch.utils.data.DataLoader(
    train_date,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_date,
    batch_size=64,
    shuffle=False
)

# train_loader va test_loader: Bu ob'ektlar train_date va test_date ma'lumotlar to'plamlarini (dataset)
# kichik qismlarga, ya'ni "batch"larga ajratish uchun ishlatiladi.
# batch_size=64: Har bir batchda 64 ta tasvir va ularning tegishli yorliqlari (label) bo'lishini bildiradi.
# shuffle=True (train_loader uchun): O'quv ma'lumotlari har bir davr (epoch) uchun tasodifiy tartibda aralashtiriladi,
# bu modelning umumlashtirish qobiliyatini yaxshilashga yordam beradi.
# shuffle=False (test_loader uchun): Test ma'lumotlari aralashtirilmaydi, chunki baholash uchun tartibning
# ahamiyati yo'q va natijalarni takrorlash osonroq bo'ladi.

In [105]:
misol = next(iter(train_loader))
misol[0].shape

# misol = next(iter(train_loader)): Bu qator train_loaderdan bitta keyingi "batch" ma'lumotni oladi.
# misol o'zgaruvchisida endi bir guruh tasvirlar va ularning tegishli yorliqlari (raqamlar) mavjud.
# misol[0].shape: misol ikki elementli ro'yxatdir, birinchi element (indeks 0) tasvirlarni o'z ichiga oladi.
# .shape esa bu tasvirlar tensorining o'lchamini (ya'ni, nechta tasvir bor, har bir tasvirning kanallari, balandligi va kengligi) ko'rsatadi.


torch.Size([64, 1, 28, 28])

# **Modelni yaratib olish**

In [106]:
class OddiyCNN(nn.Module):  # OddiyCNN klassi oddiy Konvolyutsion Neyron Tarmoq (CNN) modelini aniqlaydi:
    def __init__(self):
        super(OddiyCNN, self).__init__()

        self.net = nn.Sequential(
            # 2 blocks with Covn, Relu, Maxpool
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Flatten, Linear, relu, linear
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
      return self.net(x)

#  bu OddiyCNN klassi oddiy Konvolyutsion Neyron Tarmoq (CNN) modelini aniqlaydi:

# Konvolyutsion bloklar: Modelning boshida ikkita konvolyutsion blok bor. Har bir blok quyidagilardan iborat:
# nn.Conv2d: Kiruvchi tasvirlardan xususiyatlarni (feature) ajratib oladi.
# nn.ReLU: Nor-chiziqli faollashish funksiyasi, neyron tarmoqqa murakkablik qo'shadi.
# nn.MaxPool2d: Tasvir o'lchamini kichraytiradi va eng muhim xususiyatlarni saqlaydi.
# Yassi va to'liq bog'langan qatlamlar: Konvolyutsion qatlamlardan keyin:
# nn.Flatten: Konvolyutsion qatlamlardan chiqqan ma'lumotni bitta qatorga yassilaydi.
# nn.Linear: Bular oddiy to'liq bog'langan (fully connected) qatlamlar bo'lib, yassilangan
# xususiyatlarni tasniflash uchun ishlatiladi. Oxirgi nn.Linear qatlami 10 ta chiqishga ega,
# chunki MNIST ma'lumotlar to'plamida 0 dan 9 gacha bo'lgan 10 ta raqam mavjud.
# forward metodi: Bu modelga kiritilgan ma'lumotlar (x) qanday qilib qatlamlardan o'tishini belgilaydi.

# in_channels=1: Kiruvchi kanallar soni. Tasvir qora va oq bo'lsa (grayscale), bu odatda 1 bo'ladi. Agar rangli (RGB) tasvir bo'lsa, 3 bo'ladi.
# out_channels=16 (yoki 32): Konvolyutsion qatlamdan chiqadigan xususiyat xaritalarining (feature maps) soni.
# Bu qatlamdan keyingi neyronlar sonini belgilaydi va modelning murakkabligini oshiradi.
# kernel_size=3: Konvolyutsiya amalga oshiriladigan filtrning (yoki yadro, kernel) o'lchami. 3 degani 3x3 o'lchamdagi filtr ishlatilishini anglatadi.
# padding=1: Tasvirning chetlari atrofida qo'shiladigan nol (zero-padding) miqdori. padding=1 tasvirning chetlariga
# bitta qator nol qo'shilishini bildiradi, bu esa konvolyutsiyadan keyin tasvirning o'lchamini saqlab qolishga yordam beradi.

In [107]:
net_model = OddiyCNN()

# Bu OddiyCNN klassining yangi instansiyasini (ya'ni, aniqlangan neyron tarmoqning o'zini) yaratadi va uni
# net_model o'zgaruvchisiga tayinlaydi. Bu orqali modelni ma'lumotlar bilan ishlash va o'qitish uchun tayyor holatga keltiriladi.

In [108]:
criterion = nn.CrossEntropyLoss()
# Bu qator yo'qotish funksiyasini (loss function) belgilaydi. nn.CrossEntropyLoss asosan tasniflash vazifalari uchun
# ishlatiladi. Modelning bashoratlari (predictions) bilan haqiqiy yorliqlar (true labels) o'rtasidagi farqni hisoblash uchun xizmat qiladi.
optimizer = optim.Adam(net_model.parameters(), lr=0.001)
# Bu qator optimizatorni sozlaydi. optim.Adam – bu modelning parametrlari (vaznlar va biaslar)ni o'quv jarayonida
# yangilash uchun ishlatiladigan mashhur algoritm. net_model.parameters() modelning barcha o'rganiladigan
# parametrlarini taqdim etadi, va lr=0.001 o'rganish tezligini (learning rate) belgilaydi, ya'ni har bir
# yangilashda parametrlarning qancha miqdorda o'zgarishini boshqaradi.


In [109]:
epoxa = 10
# Model 10 marta to'liq ma'lumotlar to'plami bo'yicha o'qitiladi (har bir to'liq o'tish "epoxa" deyiladi).

net_model.to(device) # Modelni GPUga (agar mavjud bo'lsa) yoki CPUga o'tkazadi, bu hisoblashlarni tezlashtiradi.

for epox in range(epoxa):
  net_model.train()
  total_loss = 0
  for data, target in train_loader:
    data, target = data.to(device), target.to(device)

    # Forward - Oldga yuborish
    output = net_model(data)
    loss = criterion(output, target)

    # backward - Ortga yuborish
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoxa: {epox+1}, O'rtacha yo'qotish: {total_loss/len(train_loader)}")

# Asosiy tsikl (for epox in range(epoxa)): Har bir epoxa uchun quyidagi amallar takrorlanadi:
# net_model.train(): Modelni o'qitish rejimiga o'tkazadi (masalan, Dropout va BatchNorm kabi qatlamlar faollashadi).
# Ma'lumotlarni yuklash (for data, target in train_loader): Har bir o'qitish iteratsiyasida ma'lumotlarning kichik qismi (batch) yuklanadi va qurilmaga o'tkaziladi.
# output = net_model(data) (Forward): Model kiritilgan ma'lumotlardan bashorat qiladi.
# loss = criterion(output, target): Modelning bashoratlari (output) va haqiqiy yorliqlar (target) orasidagi farqni hisoblaydi.
# optimizer.zero_grad(), loss.backward(), optimizer.step() (Backward): Bu uch qadam modelning vaznlarini yangilaydi:
# Avvalgi gradientlarni tozlaydi.
# Yo'qotish funksiyasi bo'yicha gradientlarni hisoblaydi.
# Hisoblangan gradientlar asosida modelning vaznlarini yangilaydi.
# total_loss += loss.item(): Joriy batchning yo'qotishini umumiy yo'qotishga qo'shib boradi.
# print(...): Har bir epoxa oxirida o'rtacha yo'qotishni (qancha xato qilayotganini) chop etadi.
# Qisqasi, bu kod modelni belgilangan epoxalar soni bo'yicha ma'lumotlar to'plamida o'qitadi, har bir epoxada vaznlarni yangilab boradi va yo'qotishni kuzatib boradi.

Epoxa: 1, O'rtacha yo'qotish: 0.23918601592033625
Epoxa: 2, O'rtacha yo'qotish: 0.06495016729975464
Epoxa: 3, O'rtacha yo'qotish: 0.046396428354713484
Epoxa: 4, O'rtacha yo'qotish: 0.03355267084806091
Epoxa: 5, O'rtacha yo'qotish: 0.027083348749621983
Epoxa: 6, O'rtacha yo'qotish: 0.020966384343320824
Epoxa: 7, O'rtacha yo'qotish: 0.017636972249185423
Epoxa: 8, O'rtacha yo'qotish: 0.013463072014467376
Epoxa: 9, O'rtacha yo'qotish: 0.011662399356647122
Epoxa: 10, O'rtacha yo'qotish: 0.010165557094271463


In [110]:
net_model.eval()

correct = 0
total = 0

with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output = net_model(data)

    _, predicted = torch.max(output.data, 1)
    total += target.size(0)
    correct += (predicted == target).sum().item()

print(f"Test accuracy: {100 * correct / total}%")

# net_model.eval(): Modelni baholash rejimiga o'tkazadi (bu, masalan, Dropout qatlamlarini o'chirib qo'yadi,
# BatchNorm qatlamlarini esa o'qitish vaqtida hisoblangan o'rtacha va standart og'ishlardan foydalanishga majbur qiladi).
# correct = 0, total = 0: To'g'ri topilgan misollar sonini va umumiy misollar sonini hisoblash uchun o'zgaruvchilar aniqlanadi.
# with torch.no_grad():: Ushbu blok ichidagi operatsiyalarda gradientlarni hisoblashni o'chirib qo'yadi.
#  Bu test va baholash vaqtida xotira va hisoblash resurslarini tejaydi, chunki vaznlarni yangilashga hojat yo'q.
# Tsikl (for data, target in test_loader): Har bir test batchi uchun:
# Ma'lumotlar (data) va yorliqlar (target) qurilmaga (device) ko'chiriladi.
# output = net_model(data): Model kiritilgan test ma'lumotlaridan bashorat qiladi.
# _, predicted = torch.max(output.data, 1): Modelning chiqishidan eng yuqori ehtimollikka ega bo'lgan klassni (bashorat qilingan raqamni) oladi.
# total += target.size(0): Umumiy test misollari sonini yangilaydi.
# correct += (predicted == target).sum().item(): To'g'ri bashorat qilingan misollar sonini hisoblaydi.
# print(f"Test accuracy: {100 * correct / total}%"): Barcha test ma'lumotlari qayta ishlanganidan so'ng, modelning umumiy aniqligini (foizda) chop etadi.
# Qisqasi, bu kod modelni test ma'lumotlarida sinab ko'radi va u qanchalik aniqlik bilan to'g'ri javob berishini hisoblab beradi.

Test accuracy: 98.99%


biz yuqoridagi PyTorch yordamida qo'lda yozilgan raqamlarni (0 dan 9 gacha) tasniflash uchun Konvolyutsion Neyron Tarmoq (CNN) modelini yaratdik va o'qitdik.

Jarayon quyidagicha kechdi:

Ma'lumotlarni tayyorlash: Mashhur MNIST ma'lumotlar to'plami yuklab olindi. Rasmlar tensor formatiga o'tkazildi va DataLoaderlar yordamida kichik "batch"larga ajratildi, bu esa trening jarayonini samarali qildi.

Modelni yaratish: OddiyCNN nomli model aniqlandi. U bir nechta konvolyutsion qatlamlar, faollashish funksiyalari (ReLU), pooling qatlamlari, va oxirida yassi (flatten) va to'liq bog'langan (linear) qatlamlardan iborat bo'lib, tasvirlardagi xususiyatlarni ajratib olish va ularni tasniflash uchun mo'ljallangan.

Modelni o'qitish: Model Adam optimizatori va CrossEntropyLoss yo'qotish funksiyasi yordamida 10 ta "epoxa" davomida o'qitildi. Har bir epoxada model ma'lumotlarning barcha batchlari bo'yicha vaznlarini yangilab bordi va yo'qotish (loss) qiymatlari bosqichma-bosqich kamayib bordi.

Modelni baholash: Treningdan so'ng, model test ma'lumotlar to'plamida baholandi. Natijada, model 98.99% aniqlik bilan raqamlarni to'g'ri bashorat qila oldi. Bu shuni anglatadiki, model qo'lda yozilgan raqamlarni juda yuqori aniqlikda taniy olishni o'rgangan.
